### Yahan hum zaroori Spark functions import karenge aur ADF se aane wale parameters ko capture karenge.

requirements (Nulls, Validation, Quarantine, Deduplication, Casting), we will use "Split & Route" (Data Quality Rule Engine) pattern use karenge.

Is pattern me hum record ko evaluate karenge, usme ek array add karenge jo uske saare errors capture karega (jaise [NULL_PK, BAD_EMAIL]). Phir clean records ko Silver target me MERGE karenge aur bad records ko Quarantine (DLQ) table me bhejenge.

Architectural Highlights of this Logic:
Rule Engine via array_remove: Hum direct filter lagane ke bajay ek dq_violations naam ka array banate hain. Agar record me 3 galtiyan hain (e.g., Email galat hai aur Date future ki hai), toh dono rules us array me add ho jayenge. Isse data steward ko pata chalta hai ki record kis-kis wajah se fail hua.

Non-Blocking Quarantine: Bad records pipeline ko crash nahi karte. Wo silently customers_quarantine table me reject_reason ke sath save ho jate hain, jise baas me analyze ya re-process kiya ja sakta hai.

Strict Deduplication: MERGE chalane se pehle row_number() function check karta hai ki agar us run me ek hi customer_id ke multiple states aaye hain, toh sirf latest updated_at wala record target me update ho.

Dynamic Schema Merging: Quarantine table me mergeSchema true hai. Agar kal source se naya column aata hai, toh wo bina error ke quarantine aur silver target me flow karega.

In [0]:
%sql
SELECT COUNT(*) FROM dbw_atlas_dev_7405606293032023.bronze.customers;

In [0]:
%sql
-- Saari tables ki list dekhne ke liye:
SHOW TABLES IN dbw_atlas_dev_7405606293032023.bronze;

In [0]:
%sql describe dbw_atlas_dev_7405606293032023.bronze.order_items

In [0]:
%sql
select * from dbw_atlas_dev_7405606293032023.bronze.order_items

In [0]:
from pyspark.sql.functions import col, trim, lower, upper, to_timestamp, to_date, current_timestamp, row_number, when, array, array_remove, size, concat_ws, lit
from pyspark.sql.window import Window
from delta.tables import DeltaTable

# ==========================================
# 1. Variables & Setup
# ==========================================
dbutils.widgets.text("p_dataset_name", "customers")
p_dataset_name = dbutils.widgets.get("p_dataset_name")

catalog = "dbw_atlas_dev_7405606293032023"
bronze_table = f"{catalog}.bronze.{p_dataset_name}"
silver_table = f"{catalog}.silver.{p_dataset_name}"
quarantine_table = f"{catalog}.silver.{p_dataset_name}_quarantine"


In [0]:
from pyspark.sql.functions import expr 
# ==========================================
# 2. Read Bronze Data
# ==========================================
df_raw = spark.read.table(bronze_table)

# # ==========================================
# # 3. Standardize, Cleanse & Type Cast
# # ==========================================
# df_standardized = df_raw \
#     .withColumn("customer_id", trim(col("customer_id"))) \
#     .withColumn("first_name", trim(col("first_name"))) \
#     .withColumn("last_name", trim(col("last_name"))) \
#     .withColumn("email", lower(trim(col("email")))) \
#     .withColumn("phone", trim(col("phone"))) \
#     .withColumn("gender", upper(trim(col("gender")))) \
#     .withColumn("date_of_birth", to_date(col("date_of_birth"))) \
#     .withColumn("city", trim(col("city"))) \
#     .withColumn("state", trim(col("state"))) \
#     .withColumn("country", upper(trim(col("country")))) \
#     .withColumn("market", upper(trim(col("market")))) \
#     .withColumn("currency", upper(trim(col("currency")))) \
#     .withColumn("registration_date", to_timestamp(col("registration_date"))) \
#     .withColumn("customer_status", lower(trim(col("customer_status")))) \
#     .withColumn("loyalty_tier", upper(trim(col("loyalty_tier")))) \
#     .withColumn("silver_processed_timestamp", current_timestamp())

# Yeh upar imports me hona chahiye

# ==========================================
# 3. Standardize, Cleanse & Type Cast (FIXED with try_cast)
# ==========================================
df_standardized = df_raw \
    .withColumn("customer_id", trim(col("customer_id"))) \
    .withColumn("first_name", trim(col("first_name"))) \
    .withColumn("last_name", trim(col("last_name"))) \
    .withColumn("email", lower(trim(col("email")))) \
    .withColumn("phone", trim(col("phone"))) \
    .withColumn("gender", upper(trim(col("gender")))) \
    .withColumn("date_of_birth", expr("try_cast(date_of_birth as date)")) \
    .withColumn("city", trim(col("city"))) \
    .withColumn("state", trim(col("state"))) \
    .withColumn("country", upper(trim(col("country")))) \
    .withColumn("market", upper(trim(col("market")))) \
    .withColumn("currency", upper(trim(col("currency")))) \
    .withColumn("registration_date", expr("try_cast(registration_date as timestamp)")) \
    .withColumn("customer_status", lower(trim(col("customer_status")))) \
    .withColumn("loyalty_tier", upper(trim(col("loyalty_tier")))) \
    .withColumn("silver_processed_timestamp", current_timestamp())

In [0]:
df_standardized.display()

In [0]:
# # ==========================================
# # 4. Data Quality Rules Engine
# # ==========================================
# email_regex = r"^[\w\.-]+@[\w\.-]+\.\w+$"

# df_dq = df_standardized.withColumn(
#     "dq_violations",
#     array_remove(array(
#         when(col("customer_id").isNull() | (col("customer_id") == ""), lit("NULL_OR_EMPTY_CUSTOMER_ID")),
#         when(col("email").isNotNull() & ~col("email").rlike(email_regex), lit("INVALID_EMAIL_FORMAT")),
#         when(col("date_of_birth") > current_timestamp(), lit("FUTURE_DOB_ERROR")),
#         # Add more business specific rules here if needed
#     ), None)
# )

from pyspark.sql.functions import expr

# ==========================================
# 4. Data Quality Rules Engine (FIXED)
# ==========================================
email_regex = r"^[\w\.-]+@[\w\.-]+\.\w+$"

# Pehle array banayenge jisme valid errors aur Nulls dono honge
df_with_errors = df_standardized.withColumn(
    "dq_violations_raw",
    array(
        when(col("customer_id").isNull() | (trim(col("customer_id")) == ""), lit("NULL_OR_EMPTY_CUSTOMER_ID")),
        when(col("email").isNotNull() & ~col("email").rlike(email_regex), lit("INVALID_EMAIL_FORMAT")),
        when(col("date_of_birth") > current_timestamp(), lit("FUTURE_DOB_ERROR"))
    )
)

# Phir array me se Nulls ko safely hata denge
df_dq = df_with_errors.withColumn(
    "dq_violations",
    expr("filter(dq_violations_raw, x -> x is not null)")
).drop("dq_violations_raw")

# ==========================================
# 5. Split Data: Good vs Quarantine (FIXED)
# ==========================================
# Ab agar record perfect hai toh size 0 aayega, aur error hai toh > 0 aayega
df_bad = df_dq.filter(size(col("dq_violations")) > 0)
df_good = df_dq.filter(size(col("dq_violations")) == 0).drop("dq_violations")

In [0]:
# df_dq.display()

In [0]:
df_bad.display()

In [0]:
df_good.display()

In [0]:
# ==========================================
# 6. Handle QUARANTINE (Bad Records)
# ==========================================
bad_count = df_bad.count()
if bad_count > 0:
    df_bad_write = df_bad.withColumn("reject_reason", concat_ws(", ", col("dq_violations"))) \
                         .drop("dq_violations")
    
    df_bad_write.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(quarantine_table)
    print(f"Quarantined {bad_count} records to {quarantine_table}")

In [0]:
# ==========================================
# 7. Handle SILVER TARGET (Good Records)
# ==========================================
good_count = df_good.count()
if good_count > 0:
    
    # 7A. Intra-Batch Deduplication (Using ingestion_timestamp because updated_at is missing)
    window_spec = Window.partitionBy("customer_id").orderBy(col("_ingestion_timestamp").desc_nulls_last())
    
    df_deduped = df_good \
        .withColumn("row_num", row_number().over(window_spec)) \
        .filter(col("row_num") == 1) \
        .drop("row_num")
    
    # 7B. MERGE (Upsert) into Silver Table
    if spark.catalog.tableExists(silver_table):
        target_table = DeltaTable.forName(spark, silver_table)
        
        target_table.alias("target") \
            .merge(
                df_deduped.alias("source"),
                "target.customer_id = source.customer_id"
            ) \
            .whenMatchedUpdateAll() \
            .whenNotMatchedInsertAll() \
            .execute()
        print(f"MERGE successful. Processed {good_count} valid records.")
    else:
        df_deduped.write.format("delta") \
            .mode("overwrite") \
            .option("delta.autoOptimize.optimizeWrite", "true") \
            .option("delta.autoOptimize.autoCompact", "true") \
            .saveAsTable(silver_table)
        print(f"Silver table created. Inserted {good_count} valid records.")

In [0]:
# from pyspark.sql.functions import col, trim, lower, to_timestamp, current_timestamp, row_number
# from pyspark.sql.window import Window
# from delta.tables import DeltaTable

# # ADF se aane wale parameters
# dbutils.widgets.text("p_dataset_name", "customers")
# p_dataset_name = dbutils.widgets.get("p_dataset_name")

# # Database/Schema configurations
# catalog_name = "dbw_atlas_dev_7405606293032023"
# bronze_table = f"{catalog_name}.bronze.{p_dataset_name}"
# silver_table = f"{catalog_name}.silver.{p_dataset_name}"

### Is step me hum strings ko trim karenge aur timestamps ko UTC me explicitly define karenge.

In [0]:
# %sql
# select  * from dbw_atlas_dev_7405606293032023.bronze.customers

In [0]:
# %sql
# describe dbw_atlas_dev_7405606293032023.bronze.customers

In [0]:
# # 1. Read Raw Data from Bronze
# df_bronze = spark.read.table(bronze_table)

# # 2. Data Cleansing & Casting (Specific to Customers)
# if p_dataset_name == "customers":
#     df_clean = df_bronze \
#         .withColumn("first_name", trim(col("first_name"))) \
#         .withColumn("last_name", trim(col("last_name"))) \
#         .withColumn("email", lower(trim(col("email")))) \
#         .withColumn("created_at", to_timestamp(col("created_at"))) \
#         .withColumn("updated_at", to_timestamp(col("updated_at"))) \
#         .withColumn("silver_processed_timestamp", current_timestamp())

### Agar source se ek hi batch me ek customer ke do updates aa gaye, toh MERGE fail ho jayega. Yeh logic usko rokiya aur sirf latest record aage bhejega.

In [0]:
# # 3. Handle Intra-batch Duplicates
# # Assuming 'customer_id' is the primary key and 'updated_at' determines the latest state
# window_spec = Window.partitionBy("customer_id").orderBy(col("updated_at").desc(), col("_ingestion_timestamp").desc())

# df_deduped = df_clean \
#     .withColumn("row_num", row_number().over(window_spec)) \
#     .filter(col("row_num") == 1) \
#     .drop("row_num")

### Yahan hum check karenge ki agar Silver table pehle se exist karti hai toh Upsert (Merge) karo, warna pehli baar table create kar do.

In [0]:
# # 4. Check if Silver table exists, else create it
# if spark.catalog.tableExists(silver_table):
    
#     # Target (Silver) aur Source (Deduped Bronze) ko link karna
#     target_table = DeltaTable.forName(spark, silver_table)
    
#     # Merge execution with Schema Evolution enabled
#     target_table.alias("target") \
#         .merge(
#             df_deduped.alias("source"),
#             "target.customer_id = source.customer_id"  # Primary Key Join
#         ) \
#         .whenMatchedUpdateAll() \
#         .whenNotMatchedInsertAll() \
#         .execute()
        
#     print(f"MERGE successful for {p_dataset_name}")

# else:
#     # Pehli baar table ban rahi hai toh seedha Write karo
#     # Table optimization flags on kar rahe hain small files problem solve karne ke liye
#     df_deduped.write.format("delta") \
#         .mode("overwrite") \
#         .option("delta.autoOptimize.optimizeWrite", "true") \
#         .option("delta.autoOptimize.autoCompact", "true") \
#         .saveAsTable(silver_table)
        
#     print(f"Table CREATED and data inserted for {p_dataset_name}")

### Audit logs ko update karne ke liye processed rows ka count wapas bhejenge.

In [0]:
# # 5. Capture final row count for ADF logging
# try:
#     rows_processed = df_deduped.count()
# except Exception as e:
#     rows_processed = 0

# dbutils.notebook.exit(str(rows_processed))

In [0]:
# ==========================================
# 8. Return metrics to ADF
# ==========================================
total_processed = bad_count + good_count
dbutils.notebook.exit(str(total_processed))

In [0]:
%sql
SELECT customer_id, first_name, reject_reason 
FROM dbw_atlas_dev_7405606293032023.silver.customers_quarantine;